# Linear Discriminant Analysis (LDA)

Notebook นี้สาธิต Linear Discriminant Analysis สำหรับ supervised dimensionality reduction ด้วย Iris dataset.


## เป้าหมาย

- สร้าง within-class scatter $S_W$ และ between-class scatter $S_B$
- ฉายข้อมูลด้วย LDA และเปรียบเทียบกับ PCA
- ประเมิน classifier แบบป้องกัน data leakage

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

In [2]:
iris = load_iris()
X, y = iris.data, iris.target
class_names = iris.target_names
feature_names = iris.feature_names

X_scaled = StandardScaler().fit_transform(X)
pd.DataFrame(X, columns=feature_names).assign(species=[class_names[label] for label in y]).head()

## Fisher criterion

LDA เลือกทิศทาง $w$ เพื่อเพิ่มการแยก class means และลดการกระจายภายใน class:

$$J(w) = \frac{w^T S_B w}{w^T S_W w}$$

โดย $S_B$ คือ between-class scatter และ $S_W$ คือ within-class scatter

In [3]:
overall_mean = X_scaled.mean(axis=0)
S_within = np.zeros((X.shape[1], X.shape[1]))
S_between = np.zeros_like(S_within)

for class_label in np.unique(y):
    class_samples = X_scaled[y == class_label]
    class_mean = class_samples.mean(axis=0)
    centered = class_samples - class_mean
    S_within += centered.T @ centered
    difference = (class_mean - overall_mean).reshape(-1, 1)
    S_between += len(class_samples) * difference @ difference.T

print('S_W shape:', S_within.shape)
print('S_B shape:', S_between.shape)
pd.DataFrame(S_between, index=feature_names, columns=feature_names).round(2)

In [4]:
S_within

In [5]:
S_between

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
lda = LinearDiscriminantAnalysis(n_components=2)

X_pca = pca.fit_transform(X_scaled)
X_lda = lda.fit_transform(X_scaled, y)

print('PCA variance retained:', f'{pca.explained_variance_ratio_.sum():.1%}')
print('LDA shape:', X_lda.shape)
print('Maximum LDA dimensions:', len(class_names) - 1)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))
colors = ['#2563eb', '#d97706', '#059669']

for label, name, color in zip(np.unique(y), class_names, colors):
    mask = y == label
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], color=color, label=name, alpha=0.8)
    axes[1].scatter(X_lda[mask, 0], X_lda[mask, 1], color=color, label=name, alpha=0.8)

axes[0].set(title='PCA projection', xlabel='PC1', ylabel='PC2')
axes[1].set(title='LDA projection', xlabel='LD1', ylabel='LD2')
for axis in axes:
    axis.legend(frameon=False)
    axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

baseline = Pipeline([
    ('scale', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
with_lda = Pipeline([
    ('scale', StandardScaler()),
    ('lda', LinearDiscriminantAnalysis(n_components=2)),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

scores = pd.DataFrame({
    'baseline': cross_val_score(baseline, X, y, cv=folds),
    'with_lda': cross_val_score(with_lda, X, y, cv=folds),
})
scores.agg(['mean', 'std']).round(3)

In [ ]:
pipeline = Pipeline([
    ('scale', StandardScaler()),
    ('lda', LinearDiscriminantAnalysis(n_components=2)),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
pipeline.fit(X, y)

new_observation = np.array([[5.0, 3.4, 1.5, 0.2]])
prediction = pipeline.predict(new_observation)[0]
probabilities = pipeline.predict_proba(new_observation)[0]

print('Predicted species:', class_names[prediction])
pd.Series(probabilities, index=class_names, name='probability').sort_values(ascending=False)